# Aula 4, Validação

Notebook executável que acompanha a aula [04-validacao.md](../../lessons/modulo-02-fundamentos-ml/04-validacao.md).

## O que vamos fazer aqui

Como saber se um modelo é bom de verdade, e não só nos dados que viu? Vamos avaliar um
classificador de aprovação com validação cruzada de k dobras, implementada do zero, e
obter uma estimativa de desempenho mais confiável do que a de uma única divisão.

Na parte final, que é o projeto que fecha o módulo, comparamos a acurácia no treino
com a acurácia em validação, e interpretamos a diferença à luz do overfitting. O
notebook usa apenas numpy.

## Dados de aprovação

Reusamos os mesmos dados de aprovação da aula de classificação, com horas de estudo e
frequência prevendo se o aluno passa.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
n = 200
horas = rng.uniform(0, 10, size=n)
frequencia = rng.uniform(0, 1, size=n)
score = 0.6 * horas + 4 * frequencia - 4 + rng.normal(0, 1, size=n)
aprovado = (score > 0).astype(int)
X = np.column_stack([horas, frequencia])

## Validação cruzada do zero

Na validação cruzada de k dobras, dividimos os dados em k partes. Treinamos k vezes,
cada vez usando uma dobra diferente como validação e as outras como treino, e tiramos
a média dos resultados. Assim, cada exemplo serve ora para treinar, ora para validar.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def treinar_logistica(X, y, taxa=0.1, iteracoes=3000):
    w = np.zeros(X.shape[1])
    b = 0.0
    for _ in range(iteracoes):
        erro = sigmoide(X @ w + b) - y
        w -= taxa * (X.T @ erro) / len(y)
        b -= taxa * np.mean(erro)
    return w, b


def acuracia(X, y, w, b):
    previsto = (sigmoide(X @ w + b) >= 0.5).astype(int)
    return np.mean(previsto == y)


def validacao_cruzada(X, y, k=5):
    """Estima a acurácia média por validação cruzada de k dobras."""
    indices = rng.permutation(len(y))
    dobras = np.array_split(indices, k)
    resultados = []
    for j in range(k):
        idx_val = dobras[j]
        idx_tr = np.concatenate([dobras[i] for i in range(k) if i != j])
        w, b = treinar_logistica(X[idx_tr], y[idx_tr])
        resultados.append(acuracia(X[idx_val], y[idx_val], w, b))
    return np.array(resultados)


resultados = validacao_cruzada(X, aprovado, k=5)
for j, acc in enumerate(resultados, start=1):
    print(f'Dobra {j}: acurácia {acc:.2%}')
print(f'\nMédia: {resultados.mean():.2%}  |  desvio: {resultados.std():.2%}')

A média das dobras é uma estimativa mais confiável do que a de uma divisão
única, e o desvio mostra o quanto esse número é estável. Se uma dobra destoasse muito
das demais, valeria investigar se há algo peculiar naqueles exemplos.

## Projeto do módulo, treino contra validação

Este é o experimento que fecha o módulo. Vamos comparar a acurácia medida no próprio
treino, que é otimista, com a acurácia estimada por validação cruzada, que é mais
honesta, e interpretar a diferença.

In [ ]:
# Acurácia no treino completo (otimista, pois avalia onde treinou).
w, b = treinar_logistica(X, aprovado)
acc_treino = acuracia(X, aprovado, w, b)

# Acurácia estimada por validação cruzada (mais honesta).
acc_cv = validacao_cruzada(X, aprovado, k=5)

print(f'Acurácia no treino     : {acc_treino:.2%}')
print(f'Acurácia validação (cv): {acc_cv.mean():.2%}  |  desvio: {acc_cv.std():.2%}')
print()
print('A acurácia de treino tende a ser igual ou maior que a de validação.')
print('A diferença entre as duas é um indício de quanto o modelo generaliza.')

A acurácia de treino tende a ser igual ou um pouco maior que a de validação,
e essa diferença é um termômetro da generalização. Para fechar o módulo, escreva um
parágrafo comparando os dois números e ligando com o que você viu sobre overfitting.
Com isso, você percorreu o ciclo completo do aprendizado supervisionado, do modelo à
avaliação confiável.